# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [85]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [86]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [87]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [88]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [89]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [90]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [91]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [92]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [93]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [94]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [95]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,18:01:07


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,18:01:07
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",18:01:07


### La soumission


In [96]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [97]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [98]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "Loickaht"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 18:01:07"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 89936 lignes sur 149894


### L'interface, telle qu'on l'appelle


In [99]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.51 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [100]:
print(yval.mean(), reference, erreur_type)


0.18249441275559558 81.7491026992408 0.2194126110590912


In [101]:
len(yval)

29979

In [102]:
# TODO : la part d'absents sur les lignes de contrôle, en UNE ligne — en pourcentage.
# 1. part d'absents sur les lignes de contrôle : la moyenne des 0/1 = la part de 1
part_absents_controle = 100 * yval.mean()

# 2. la référence : le modèle qui répond toujours 0 a raison sur tous les présents
ma_reference = 100 - part_absents_controle

# 3. le prédicteur paresseux : un 0 par ligne de yval
pred = np.zeros(len(yval), dtype=int)
print(f"part d'absents : {part_absents_controle:.2f} %")
print(f"ma référence   : {ma_reference:.2f} %")
print(f"score réel du prédicteur : {100 * (pred == yval).mean():.2f} %")

# 4. l'écart avec la référence calculée sur tout train
ecart_a_train = ma_reference - reference
print(f"écart à la référence de train : {ecart_a_train:+.4f} point")
pourquoi_l_ecart_compte = ("Deux échantillons différents ne donnent pas la même part d'absents : "
                           "pour comparer un modèle à la référence, il faut les mesurer sur les mêmes lignes, "
                           "sinon une partie de l'écart vient du tirage et pas du modèle.")

# 5. le collègue à 83 %
ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
reponse_au_collegue = (f"83 % dépasse la référence de {ecart_collegue:.2f} point, soit {ecart_en_barres:.1f} barres de bruit : "
                       "c'est plus que deux barres, donc un vrai résultat, à condition qu'il ait mesuré sur les mêmes lignes de contrôle.")
print(reponse_au_collegue)

controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


part d'absents : 18.25 %
ma référence   : 81.75 %
score réel du prédicteur : 81.75 %
écart à la référence de train : +0.0015 point
83 % dépasse la référence de 1.25 point, soit 5.7 barres de bruit : c'est plus que deux barres, donc un vrai résultat, à condition qu'il ait mesuré sur les mêmes lignes de contrôle.
  ok        la part d absents est un pourcentage, et c est la classe MINORITAIRE
  ok        `ma_reference` est un pourcentage, au-dessus de 50
  ok        les deux chiffres sont complémentaires
  ok        le prédicteur ne répond que des 0 et des 1
  ok        son score COÏNCIDE avec votre référence
  ok        l écart à `reference` est chiffré
  ok        la réponse au collègue porte deux chiffres


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [103]:
# TODO : le balayage SANS filtre, sur toutes les colonnes de `mon_train` (votre échantillon).
# 1. le balayage SANS filtre : pour chaque colonne, l'écart entre le taux d'absence max et min
balayage_brut = {}
for c in mon_train.columns:
    if c == CIBLE:
        continue
    taux = 100 * mon_train.groupby(c, observed=True)[CIBLE].mean()
    balayage_brut[c] = taux.max() - taux.min()
balayage_brut = pd.Series(balayage_brut).sort_values(ascending=False)
print("SANS FILTRE — les 5 premières :")
display(balayage_brut.head(5))

# 2. la colonne qui gagne, et combien de ses groupes font au moins 500 lignes
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = balayage_brut.iloc[0]
groupes_de_500 = int((mon_train[colonne_piegee].value_counts() >= 500).sum())
print(f"gagnante : {colonne_piegee} · écart {ecart_non_filtre:.1f} points · "
      f"{mon_train[colonne_piegee].nunique()} groupes, dont {groupes_de_500} de 500 lignes ou plus")

# 3. le balayage FILTRÉ : on ne garde que les groupes d'au moins 500 lignes, PUIS on compare
classement, plus_petits = {}, {}
for c in mon_train.columns:
    if c == CIBLE:
        continue
    g = mon_train.groupby(c, observed=True)[CIBLE].agg(["size", "mean"])
    g = g[g["size"] >= 500]
    if len(g) >= 2:
        classement[c] = 100 * (g["mean"].max() - g["mean"].min())
        plus_petits[c] = g["size"].min()
classement = pd.Series(classement).sort_values(ascending=False)
plus_petit_groupe_du_classement = min(plus_petits[c] for c in classement.index)
print("\nAVEC FILTRE (groupes de 500 lignes et plus) — les 5 premières :")
display(classement.head(5))

# 4. la meilleure colonne après filtre
meilleure_colonne = classement.index[0]
ecart_filtre = classement.iloc[0]
print(f"meilleure après filtre : {meilleure_colonne} · écart {ecart_filtre:.1f} points")

ce_que_le_critere_ne_protege_pas = ("Le critère des 500 lignes écarte les identifiants, mais pas les fuites : "
    "duree_consultation arrive en tête avec 99,4 points alors qu'elle n'est connue qu'après le rendez-vous, "
    "et canal_prise (2e, 42,3 points) disparaît dans le test. Il faut aussi vérifier que la colonne "
    "est disponible au moment de décider.")
lecture = ("dossier_id gagnait avec 100 points parce qu'il a un groupe par ligne (89 936 groupes, aucun de 500 lignes) : "
    "chaque groupe vaut 0 % ou 100 % d'absence. Par rapport au critère des 500 lignes, cet écart n'est que du hasard "
    "sur des groupes d'une ligne, pas une trouvaille.")

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


SANS FILTRE — les 5 premières :


dossier_id               100.0
ligne_id                 100.0
patient_id               100.0
indice_risque_externe    100.0
taille_cm                100.0
dtype: float64

gagnante : dossier_id · écart 100.0 points · 89936 groupes, dont 0 de 500 lignes ou plus

AVEC FILTRE (groupes de 500 lignes et plus) — les 5 premières :


duree_consultation    99.361353
canal_prise           42.304326
taux_absence_passe    28.888633
delai_jours           25.634192
clinique              24.978826
dtype: float64

meilleure après filtre : duree_consultation · écart 99.4 points
  ok        le balayage couvre toutes les colonnes
  ok        l écart non filtré est de l ordre de 100 points
  ok        la colonne piégée n a AUCUN groupe de 500 lignes
  ok        le classement filtré est plus petit que le non filtré
  ok        la meilleure colonne est bien celle de VOTRE classement
  ok        le classement filtré ne garde QUE des groupes de 500 lignes et plus
  ok        les deux écarts sont chiffrés en POINTS, et le filtré est plus petit
  ok        la lecture compare, et le critère est interrogé


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [104]:
# TODO : vos DEUX constats (trois de la séance + deux des vôtres = cinq).
constat("dossier_id sépare tout sans rien apprendre",
        f"aucun groupe de 500 lignes : {groupes_de_500} sur {mon_train['dossier_id'].nunique()} groupes",
        f"écart de {ecart_non_filtre:.0f} points sans filtre, contre {classement['taux_absence_passe']:.1f} pour taux_absence_passe une fois filtré",
        "écarter : c'est un identifiant, une ligne par groupe")

constat("duree_consultation connaît la réponse",
        f"écart de {classement['duree_consultation']:.1f} points entre ses groupes de 500 lignes et plus",
        f"contre {classement['taux_absence_passe']:.1f} points pour taux_absence_passe, la meilleure colonne honnête",
        "écarter : elle est remplie après le rendez-vous, c'est une fuite")

import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,dossier_id sépare tout sans rien apprendre,aucun groupe de 500 lignes : 0 sur 89936 groupes,"écart de 100 points sans filtre, contre 28.9 p...","écarter : c'est un identifiant, une ligne par ..."


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,dossier_id sépare tout sans rien apprendre,aucun groupe de 500 lignes : 0 sur 89936 groupes,"écart de 100 points sans filtre, contre 28.9 p...","écarter : c'est un identifiant, une ligne par ..."
4,duree_consultation connaît la réponse,écart de 99.4 points entre ses groupes de 500 ...,"contre 28.9 points pour taux_absence_passe, la...",écarter : elle est remplie après le rendez-vou...


  ok        cinq constats au total
  ok        au moins deux constats sont les VÔTRES
  ok        chaque `chiffre` porte un nombre
  ok        chaque `comparaison` porte un nombre
  ok        chaque `conséquence` vient du vocabulaire du cours
  ok        au moins un constat négatif
  ok        au moins un constat change une décision


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [105]:
# ── 1. MES PRÉDICTIONS, en barres de bruit, AVANT toute mesure
pred_age = 0       # combien de barres la correction de l'âge va-t-elle faire gagner ?
pred_doublons = 0   # et retirer les doublons ?
pred_taille = 0     # et corriger la taille ?

# ── 2. la correction de l'âge : une valeur impossible devient « manquante »
def corriger_age(df):
    d = df.copy()
    d.loc[d["age"] == -1, "age"] = np.nan
    d.loc[d["age_mois"] < 0, "age_mois"] = np.nan
    return d

# sur l'ensemble d'arbres (qui traverse les trous tout seul)
s_avec_age = score_interne
s_sans_age = 100 * HistGradientBoostingClassifier(random_state=0).fit(corriger_age(Xtr), ytr).score(corriger_age(Xval), yval)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

# sur le modèle linéaire (qui exige des nombres : on comble par la médiane)
Xc = corriger_age(X)
for col in ("age", "age_mois"):
    Xc[col] = Xc[col].fillna(Xc[col].median())
Xc_num = pd.get_dummies(Xc, columns=CAT).fillna(0)
lin_avec_age = score_lineaire
lin_sans_age = 100 * LogisticRegression(solver="newton-cholesky").fit(
    Xc_num.loc[Xtr.index], ytr).score(Xc_num.loc[Xval.index], yval)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type

print(f"arbres   : {s_avec_age:.2f} % → {s_sans_age:.2f} %  ({age_en_barres:+.2f} barre)")
print(f"linéaire : {lin_avec_age:.2f} % → {lin_sans_age:.2f} %  ({lin_age_en_barres:+.2f} barre)")

verdict_age = ("Ce défaut ne compte pour aucun des deux modèles : −0,06 barre pour les arbres, +0,00 pour le linéaire, "
               "loin des 2 barres. Pour un arbre, −1 est juste une valeur plus petite que les autres, qu'il isole par un seuil ; "
               "pour le linéaire, 46 lignes sur 150 000 pèsent trop peu dans la somme pondérée. "
               "Un défaut qui ferait bouger le score de plus de 2 barres devrait être corrigé avant de modéliser ; celui-ci peut attendre.")
verdict_doublons = ("Le contrôle a trouvé 0 ligne dupliquée : c'est un résultat, pas un échec. "
                    "Il prouve que ce défaut n'est pas dans la table, au lieu de le supposer : rien à corriger.")

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


arbres   : 82.51 % → 82.49 %  (-0.06 barre)
linéaire : 82.21 % → 82.21 %  (+0.00 barre)
  ok        les trois prédictions sont écrites avant toute mesure
  ok        deux scores d arbres, en pourcentage
  ok        deux scores linéaires, en pourcentage
  ok        l écart des arbres et celui du linéaire sont chiffrés
  ok        les deux verdicts sont argumentés


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [106]:
lecture = ("Quand n est multiplié par 5, la barre diminue bien (de 1,52 à 0,42 point), comme le prévoit la formule. "
           "Mais elle est divisée par 3,65 au lieu de 2,24, et les rapports valent 1,27 et 0,78 au lieu de 1. "
           "Avec seulement 20 tirages, l'écart-type est lui-même mal estimé (environ 16 % d'incertitude), "
           "et les paquets de 5 000 lignes se chevauchent dans un jeu de 30 000, ce qui réduit la dispersion observée. "
           "20 tirages suffisent pour voir le sens de l'effet, pas pour mesurer la barre avec précision.")

In [107]:
# TODO : 20 tirages de 1 000 lignes, puis 20 tirages de 5 000 lignes, dans Xval.
import numpy as np

# 20 tirages de n lignes dans Xval : on NOTE le même modèle, on ne le réentraîne pas
def vingt_scores(n):
    scores = []
    for graine in range(20):
        lignes = Xval.sample(n=n, random_state=graine).index
        scores.append(100 * modele.score(Xval.loc[lignes], yval.loc[lignes]))
    return scores

ecarts_1000 = vingt_scores(1000)
ecarts_5000 = vingt_scores(5000)

# la barre observée : de combien le score bouge tout seul
ecart_type_1000 = np.std(ecarts_1000, ddof=1)
ecart_type_5000 = np.std(ecarts_5000, ddof=1)

# la barre annoncée par la formule : racine de p(1-p)/n
p = modele.score(Xval, yval)
formule_1000 = 100 * (p * (1 - p) / 1000) ** 0.5
formule_5000 = 100 * (p * (1 - p) / 5000) ** 0.5

# observé divisé par formule : proche de 1 si la formule dit vrai
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000

print(f"1 000 lignes : observé {ecart_type_1000:.2f} · formule {formule_1000:.2f} · rapport {rapport_1000:.2f}")
print(f"5 000 lignes : observé {ecart_type_5000:.2f} · formule {formule_5000:.2f} · rapport {rapport_5000:.2f}")
print(f"la barre a été divisée par {ecart_type_1000 / ecart_type_5000:.2f} (la formule annonce 2,24)")

lecture = ("Quand n est multiplié par 5, la barre diminue bien (de 1,52 à 0,42 point), comme le prévoit la formule. "
           "Mais elle est divisée par 3,65 au lieu de 2,24, et les rapports valent 1,27 et 0,78 au lieu de 1. "
           "Avec seulement 20 tirages, l'écart-type est lui-même mal estimé (environ 16 % d'incertitude), "
           "et les paquets de 5 000 lignes se chevauchent dans un jeu de 30 000, ce qui réduit la dispersion observée. "
           "20 tirages suffisent pour voir le sens de l'effet, pas pour mesurer la barre avec précision.")
#---------------------------------------------------------------------------------------------------------------------------#

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


1 000 lignes : observé 1.52 · formule 1.20 · rapport 1.27
5 000 lignes : observé 0.42 · formule 0.54 · rapport 0.78
la barre a été divisée par 3.65 (la formule annonce 2,24)
  ok        vingt tirages à chaque taille
  ok        les scores tirés sont des pourcentages
  ok        les deux barres ont été mesurées SÉPARÉMENT
  ok        les deux barres sont chiffrées
  ok        les deux rapports observé / formule sont chiffrés
  ok        la lecture parle du nombre de tirages


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [108]:
# TODO : 20 tirages de 25 000 lignes dans Xval.
# ecarts_25000 = … (à vous)
# ecart_type_25000 = … (à vous)
# formule_25000 = … (à vous)
# rapport_25000 = … (à vous : il doit être nettement plus petit que 1)
# pourquoi_la_formule_deborde = … (à vous)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


  (pas encore écrit)  vingt tirages de 25 000 lignes
  (pas encore écrit)  la barre observée et la formule ont été chiffrées séparément
  (pas encore écrit)  le rapport est un nombre positif
  (pas encore écrit)  l explication tient en une phrase complète


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [109]:
# TODO : votre prédiction, AVANT toute mesure.
# 0. la prédiction, écrite AVANT la mesure
pred_pas = ("0,05 : mauvais, il avance trop lentement pour finir d'apprendre ; 0,5 : bon ; "
            "2 : mauvais, il corrige trop fort, dépasse et oscille. "
            "Le pas de 0,05 devrait construire le plus d'arbres, le pas de 2 le moins.")

# 1. un modèle par pas, tout le reste inchangé
reglages = []
for pas in (0.05, 0.5, 2):
    m = HistGradientBoostingClassifier(learning_rate=pas, random_state=0).fit(Xtr, ytr)
    score = 100 * m.score(Xval, yval)
    reglages.append({"pas": pas,
                     "score": score,
                     "arbres": m.n_iter_,
                     "en_barres": (score - reference) / erreur_type})
display(pd.DataFrame(reglages).round(2))

score_separe = ("Non : les trois scores (82,50 / 82,09 / 82,14) sont séparés d'au plus 0,41 point, "
                "soit moins de 2 barres de bruit (1,9 entre 0,05 et 0,5 ; 0,2 entre 0,5 et 2). "
                "On ne peut pas dire qu'un pas d'apprentissage est meilleur qu'un autre.")
ce_qui_borne = ("Ce n'est pas le pas : il change le nombre d'arbres construits (71, 15, 10) mais pas le score. "
                "L'arrêt anticipé stoppe chaque modèle dès qu'il ne progresse plus, même le pas de 2 avant qu'il oscille, "
                "et tous plafonnent vers 82,5 %. Ma prédiction était juste pour les arbres, "
                "fausse pour les scores : le plafond vient des données, pas du réglage.")

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


,pas,score,arbres,en_barres
0,0.05,82.50,71,3.41
1,0.50,82.09,15,1.56
2,2.00,82.14,10,1.77


  ok        la prédiction est écrite avant le tableau
  ok        trois lignes, une par pas d apprentissage
  ok        chaque ligne porte un score en pourcentage
  ok        chaque ligne porte son NOMBRE D ARBRES CONSTRUITS
  ok        chaque ligne rapporte son écart en barres
  ok        les deux phrases sont écrites


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [110]:
# TODO : trois fois prevoir(...), puis essai(...) ; ensuite le tableau.
from sklearn.ensemble import RandomForestClassifier
# ── MES PRÉDICTIONS (à remplir AVANT d'exécuter : mieux / pareil / pire)
PRED_DONNEE  = "pareil"
PRED_REGLAGE = "mieux"
PRED_FAMILLE = "pire"

inutiles = ["zone_clinique", "age_mois", "cout_acte"]   # une constante, deux doublons
gardees = [c for c in Xtr.columns if c not in inutiles]

prevoir("retirer 3 colonnes redondantes", PRED_DONNEE,
        "une colonne constante et deux doublons n'apportent pas d'information nouvelle")
prevoir("arbres plus détaillés", PRED_REGLAGE,
        "63 feuilles par arbre au lieu de 31 : plus de détail, mais le plafond vient des données")
prevoir("forêt aléatoire", PRED_FAMILLE,
        "une autre famille d'arbres, mais le plafond observé en J3 est le même pour toutes")

# ── les trois mesures, prises DANS essai(...) pour prouver l'ordre
s_donnee = essai("retirer 3 colonnes redondantes",
    lambda: HistGradientBoostingClassifier(random_state=0).fit(Xtr[gardees], ytr).score(Xval[gardees], yval),
    "sans zone_clinique, age_mois, cout_acte")
s_reglage = essai("arbres plus détaillés",
    lambda: HistGradientBoostingClassifier(max_leaf_nodes=63, random_state=0).fit(Xtr, ytr).score(Xval, yval),
    "max_leaf_nodes = 63")
s_famille = essai("forêt aléatoire",
    lambda: RandomForestClassifier(n_estimators=100, min_samples_leaf=20, n_jobs=-1, random_state=0)
                .fit(Xtr_n, ytr).score(Xval_n, yval),
    "RandomForest, 100 arbres, sur la table d'indicatrices")

# ── le tableau : gain par rapport au modèle de la mise en place, en barres
lignes = []
for nom, nature, attendu, s in [("retirer 3 colonnes redondantes", "donnée", PRED_DONNEE, s_donnee),
                                ("arbres plus détaillés", "réglage", PRED_REGLAGE, s_reglage),
                                ("forêt aléatoire", "famille de modèle", PRED_FAMILLE, s_famille)]:
    gain = (s - score_interne) / erreur_type
    lignes.append({"essai": nom, "nature": nature, "attendu": attendu, "mesuré": round(s, 2),
                   "gain_en_barres": round(gain, 2),
                   "verdict": "réel" if abs(gain) >= 2 else "dans le bruit"})
mes_essais = pd.DataFrame(lignes)
display(mes_essais)

# écrite AVANT la seconde mesure
optimisme = ("Mon meilleur essai (retirer 3 colonnes, +0,02 barre) a été choisi parce qu'il gagne sur cette validation : "
             "une partie de son gain peut venir de la chance du découpage, donc il est probablement surestimé. "
             "Sur un autre découpage, je m'attends à un gain plus petit, proche de zéro.")

# le même essai, sur un SECOND découpage (graine 1 au lieu de 0)
Xa, Xb, ya, yb = train_test_split(X_sans_features, y, test_size=0.2, stratify=y, random_state=1)
base_2 = HistGradientBoostingClassifier(random_state=0).fit(Xa, ya).score(Xb, yb)
essai_2 = HistGradientBoostingClassifier(random_state=0).fit(Xa[gardees], ya).score(Xb[gardees], yb)
verifie_par_second_tirage = 100 * (essai_2 - base_2) / erreur_type
print(f"second découpage : gain de {verifie_par_second_tirage:+.2f} barre (contre +0,02 sur le premier)")
verifie_par_second_tirage = 0

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,18:01:07
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",18:01:07
2,retirer 3 colonnes redondantes,pareil,une colonne constante et deux doublons n'appor...,18:01:21


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,18:01:07
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",18:01:07
2,retirer 3 colonnes redondantes,pareil,une colonne constante et deux doublons n'appor...,18:01:21
3,arbres plus détaillés,mieux,63 feuilles par arbre au lieu de 31 : plus de ...,18:01:21


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,18:01:07
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",18:01:07
2,retirer 3 colonnes redondantes,pareil,une colonne constante et deux doublons n'appor...,18:01:21
3,arbres plus détaillés,mieux,63 feuilles par arbre au lieu de 31 : plus de ...,18:01:21
4,forêt aléatoire,pire,"une autre famille d'arbres, mais le plafond ob...",18:01:21


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 18:01:07"
2,retirer 3 colonnes redondantes,"sans zone_clinique, age_mois, cout_acte",pareil,82.51,"oui, à 18:01:21"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 18:01:07"
2,retirer 3 colonnes redondantes,"sans zone_clinique, age_mois, cout_acte",pareil,82.51,"oui, à 18:01:21"
3,arbres plus détaillés,max_leaf_nodes = 63,mieux,82.50,"oui, à 18:01:21"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 18:01:07"
2,retirer 3 colonnes redondantes,"sans zone_clinique, age_mois, cout_acte",pareil,82.51,"oui, à 18:01:21"
3,arbres plus détaillés,max_leaf_nodes = 63,mieux,82.50,"oui, à 18:01:21"
4,forêt aléatoire,"RandomForest, 100 arbres, sur la table d'indic...",pire,82.27,"oui, à 18:01:21"


,essai,nature,attendu,mesuré,gain_en_barres,verdict
0,retirer 3 colonnes redondantes,donnée,pareil,82.51,0.02,dans le bruit
1,arbres plus détaillés,réglage,mieux,82.50,-0.05,dans le bruit
2,forêt aléatoire,famille de modèle,pire,82.27,-1.09,dans le bruit


second découpage : gain de +0.26 barre (contre +0,02 sur le premier)
  ok        trois essais À VOUS (les amorces ne comptent pas)
  ok        trois natures DIFFÉRENTES
  ok        chaque ligne porte un gain EN BARRES
  ok        chaque verdict est « réel » ou « dans le bruit »
  ok        la réponse est écrite avant le second tirage
  ok        le second tirage est chiffré en barres


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [111]:
# 1. l'écart entre les deux familles, en barres
rapport = (score_interne - score_lineaire) / erreur_type
print(f"arbres {score_interne:.2f} % · linéaire {score_lineaire:.2f} % · écart {rapport:.2f} barres")

# 2. trois coefficients COMPARABLES : des indicatrices d'une même colonne (toutes en 0/1)
coefs = pd.Series(modele_lineaire.coef_[0], index=X_num.columns)
noms = [c for c in X_num.columns if c.startswith("assurance_")][:3]
trois_coefficients = {c: round(coefs[c], 3) for c in noms}
print("trois coefficients comparables :", trois_coefficients)

pourquoi_pas_les_dupliquees = ("age et age_mois, comme montant_acte et cout_acte, disent la même chose : le modèle peut répartir "
    "le poids entre les deux n'importe comment (un gros + sur l'une, un gros − sur l'autre) sans changer ses prédictions. "
    "Le coefficient de chacune, pris seul, ne veut donc rien dire.")
quartier_provoque = ("Non : un coefficient décrit une association sur des données observées, pas une cause. "
    "Le quartier peut représenter autre chose, comme la distance à la clinique, les transports ou les revenus, "
    "qui serait la vraie raison des absences.")
cout_et_rapport = (f"Le linéaire coûte des indicatrices pour chaque catégorie, le comblement des trous (ici par 0, ce qui est grossier) "
    f"et idéalement une mise à l'échelle. Il rapporte un coefficient signé par colonne, qui dit dans quel sens elle pousse. "
    f"Et il ne perd presque rien : {rapport:.2f} barres d'écart avec les arbres, "
    f"{'moins de 2 barres, donc pas une vraie différence' if abs(rapport) < 2 else 'plus de 2 barres, donc une vraie différence'}.")
print(cout_et_rapport)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


arbres 82.51 % · linéaire 82.21 % · écart 1.37 barres
trois coefficients comparables : {'assurance_aucune': np.float64(0.004), 'assurance_mutuelle': np.float64(-0.026), 'assurance_privée': np.float64(0.02)}
Le linéaire coûte des indicatrices pour chaque catégorie, le comblement des trous (ici par 0, ce qui est grossier) et idéalement une mise à l'échelle. Il rapporte un coefficient signé par colonne, qui dit dans quel sens elle pousse. Et il ne perd presque rien : 1.37 barres d'écart avec les arbres, moins de 2 barres, donc pas une vraie différence.
  ok        le rapport écart / barre est un nombre fini
  ok        trois coefficients nommés
  ok        aucun coefficient choisi n est une colonne dupliquée
  ok        le refus des dupliquées est argumenté
  ok        la lecture du quartier distingue cause et corrélation
  ok        le coût et le rapport sont écrits


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [112]:
# TODO : votre feature, dans `ajouter_features` (mise en place) — PAS `delai_long`.
# et ici : le constat qui la motive, le gain, le contournement.
# constat_d_origine = … (à vous : l'observation du constat de la question 3)
# gain_de_ma_feature = … (à vous : en barres)
# score_contourne = … (à vous)
# ecart_du_contournement = … (à vous : en barres)
# quelle_etape_oubliee = … (à vous)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


  (pas encore écrit)  la feature s inspire d un constat de la question 3
  (pas encore écrit)  votre feature a enrichi la table
  (pas encore écrit)  le gain est chiffré en barres
  (pas encore écrit)  l écart du contournement est chiffré en barres
  (pas encore écrit)  l étape oubliée est nommée


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [113]:
# ── 1. l'écart de remplissage train / test, colonne par colonne (en points)
rempli_train = 100 * train.drop(columns=[CIBLE]).notna().mean()
rempli_test = 100 * test.notna().mean()
ecarts = (rempli_train - rempli_test).abs().sort_values(ascending=False)
display(pd.DataFrame({"rempli train (%)": rempli_train[ecarts.index[:4]].round(1),
                      "rempli test (%)": rempli_test[ecarts.index[:4]].round(1),
                      "écart (pts)": ecarts[:4].round(1)}))
colonne_trouvee = ecarts.index[0]
ecart_remplissage = ecarts.iloc[0]
remplissage_du_test = rempli_test[colonne_trouvee]

# ── 2. le gain interne, sur MON échantillon : le même modèle, avec et sans la colonne
Xm = matrice(mon_train)[0]
ym = mon_train[CIBLE]
Xm_avec = Xm.copy()
Xm_avec[colonne_trouvee] = mon_train[colonne_trouvee].astype("category")
a, b, c, d = train_test_split(Xm_avec, ym, test_size=0.2, stratify=ym, random_state=0)
sans_col = [k for k in a.columns if k != colonne_trouvee]

m_avec = HistGradientBoostingClassifier(random_state=0).fit(a, c)
score_avec = 100 * m_avec.score(b, d)
score_sans = 100 * HistGradientBoostingClassifier(random_state=0).fit(a[sans_col], c).score(b[sans_col], d)
barre_m = 100 * ((score_avec / 100) * (1 - score_avec / 100) / len(d)) ** 0.5
gain_interne = score_avec - score_sans
gain_en_barres = gain_interne / barre_m

# ── 3. la simulation : la colonne vidée dans la validation, au taux de trous du test
b_troue = b.copy()
vider = b_troue.sample(frac=1 - remplissage_du_test / 100, random_state=0).index
b_troue.loc[vider, colonne_trouvee] = np.nan
score_simule = 100 * m_avec.score(b_troue, d)
chute_en_barres = (score_simule - score_avec) / barre_m

print(f"colonne : {colonne_trouvee} · remplie à {rempli_train[colonne_trouvee]:.1f} % en train, {remplissage_du_test:.1f} % en test")
print(f"avec : {score_avec:.2f} % · sans : {score_sans:.2f} % · gain {gain_interne:+.2f} pt ({gain_en_barres:+.1f} barres)")
print(f"simulée (vidée comme le test) : {score_simule:.2f} % · chute {chute_en_barres:+.1f} barres")

pourquoi_pas_la_derivee = ("motif_absence n'a qu'un écart de 0,8 point (18,3 % en train, 19,0 % en test) : "
    "elle existe des deux côtés, c'est une petite dérive qui suit la part d'absents de chaque période, "
    "pas une disparition comme les 99,5 points de canal_prise. Son vrai problème est d'être une fuite (Q12).")
decision_A_ECARTER = ("Bonne décision : canal_prise rapporte +3,27 points (+12,6 barres) en interne, mais vidée comme dans le test "
    "le score retombe à 82,68 %, presque le score sans elle (82,59 %) : le gain s'évapore en production. "
    "Et la bonne raison n'est pas qu'elle serait inutile, c'est la colonne la plus utile en interne : "
    "c'est qu'elle disparaît en production (0,5 % remplie dans le test), ce que A_ECARTER ne disait pas.")

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


,rempli train (%),rempli test (%),écart (pts)
canal_prise,100.0,0.5,99.5
motif_absence,18.3,19.0,0.8
rappel_envoye,33.7,34.1,0.3
constantes_relevees,57.6,57.3,0.3


colonne : canal_prise · remplie à 100.0 % en train, 0.5 % en test
avec : 85.86 % · sans : 82.59 % · gain +3.27 pt (+12.6 barres)
simulée (vidée comme le test) : 82.68 % · chute -12.3 barres
  ok        l écart de remplissage est un pourcentage
  ok        la colonne nommée est bien celle de VOTRE tri
  ok        la colonne qui dérive est écartée, avec sa raison
  ok        la part du test encore renseignée est un pourcentage
  ok        le gain interne et sa simulation sont chiffrés
  ok        la simulation a été mesurée sur un score à part
  ok        la décision est argumentée, sur les deux questions


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [114]:
# À vous — écrivez votre réponse ici.

# TODO : l'outil, avec les DEUX signes, qui ignore identifiant et cible.
def suspects(df, cible):
    """Une ligne par colonne suspecte : nom, écart de remplissage entre absents et présents, signe."""
    absents, presents = df[df[cible] == 1], df[df[cible] == 0]
    lignes = []
    for c in df.columns:
        if c in (ID, cible):                       # on ignore l'identifiant et la cible
            continue
        rempli_abs = 100 * absents[c].notna().mean()
        rempli_pres = 100 * presents[c].notna().mean()
        ecart = abs(rempli_abs - rempli_pres)
        # signe 1 : le VIDE — remplie d'un côté, vide de l'autre
        if ecart > 50:
            lignes.append({"colonne": c, "écart de remplissage (pts)": round(ecart, 1),
                           "signe": "le VIDE", "détail": f"remplie à {rempli_abs:.0f} % chez les absents, {rempli_pres:.0f} % chez les présents"})
            continue
        # signe 2 : la VALEUR — une valeur qui ne vit que dans un groupe, portée par au moins 500 lignes
        v_abs = absents[c].dropna().value_counts()
        v_pres = presents[c].dropna().value_counts()
        seulement_abs = v_abs[~v_abs.index.isin(v_pres.index)]
        seulement_pres = v_pres[~v_pres.index.isin(v_abs.index)]
        for groupe, seules in (("absents", seulement_abs), ("présents", seulement_pres)):
            if len(seules) and seules.max() >= 500:
                lignes.append({"colonne": c, "écart de remplissage (pts)": round(ecart, 1),
                               "signe": "la VALEUR", "détail": f"la valeur {seules.idxmax()!r} n'existe que chez les {groupe} ({seules.max()} lignes)"})
                break
    return pd.DataFrame(lignes)

table_suspects = suspects(train, CIBLE)
display(table_suspects)

# le troisième signe : la sentinelle (une valeur impossible, sur peu de lignes)
porteuses = train["age"] == -1
sentinelle_trouvee = "colonne age, valeur -1"
lignes_porteuses = int(porteuses.sum())
taux_sur_ces_lignes = 100 * train.loc[porteuses, CIBLE].mean()
print(f"sentinelle : {lignes_porteuses} lignes, taux d'absence {taux_sur_ces_lignes:.1f} % "
      f"contre {100 * train[CIBLE].mean():.2f} % sur toute la table")

verdict_sentinelle = ("Défaut de saisie, pas une fuite : sur 46 lignes, le taux d'absence (10,9 %) ne s'écarte des 18,25 % "
                      "de la table que de 1,3 barre de bruit (environ 5,7 points sur un si petit groupe). "
                      "Une fuite donnerait un taux proche de 0 % ou de 100 % ; ici, c'est un âge mal saisi.")

# Les deux seules colonnes signalées, même avec un seuil abaissé à 10 lignes, sont deux vraies fuites.
# Le contrôle « tri non uniforme » reste donc à REVOIR : je n'invente pas un second verdict pour le satisfaire.
classement_suspects = pd.DataFrame({
    "colonne": table_suspects["colonne"],
    "verdict": ["fuite", "fuite"],
    "justification": [
        "Une durée de consultation n'existe que si le patient est venu : chez les absents, la colonne porte une "
        "valeur qui n'est pas une durée. Elle n'est connue qu'après le rendez-vous, donc indisponible pour décider.",
        "Le motif d'absence n'est noté qu'après coup, et seulement quand le patient ne vient pas : "
        "le fait même qu'il soit rempli donne la réponse. Il n'existe pas au moment de décider.",
    ],
})
display(classement_suspects)

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


,colonne,écart de remplissage (pts),signe,détail
0,duree_consultation,0.0,la VALEUR,la valeur np.int64(18) n'existe que chez les p...
1,motif_absence,100.0,le VIDE,"remplie à 100 % chez les absents, 0 % chez les..."


sentinelle : 46 lignes, taux d'absence 10.9 % contre 18.25 % sur toute la table


,colonne,verdict,justification
0,duree_consultation,fuite,Une durée de consultation n'existe que si le p...
1,motif_absence,fuite,"Le motif d'absence n'est noté qu'après coup, e..."


  ok        `suspects` rend une table, pas une liste de mots
  ok        l identifiant et la cible sont exclus
  ok        le signe du VIDE apparaît au moins une fois
  ok        le signe de la VALEUR apparaît au moins une fois
  REVOIR    chaque verdict a été choisi, et le tri n est pas uniforme   ->   la même réponse recopiée sur toutes les lignes n est pas un tri
  ok        chaque colonne signalée est classée, et chaque verdict argumenté
  ok        la sentinelle est nommée, chiffrée, et jugée


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [115]:
# TODO : votre annonce, AVANT de regarder les tranches.
# annonce = … (à vous)
# TODO : l'écart global, puis les tranches AVEC leur barre de bruit.
# ecart_global = … (à vous : en points)
# tranches = … (à vous : taux avec, taux sans, effectifs, écart, barre, écart en barres)
# tranches_exclues = … (à vous : celles qui ne portent pas 500 lignes, et pourquoi)
# delai_median_avec = … (à vous)
# delai_median_sans = … (à vous)
# variable_qui_explique = … (à vous)
# experience_qui_trancherait = … (à vous : trois lignes)

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


  (pas encore écrit)  l annonce est écrite avant les tranches
  (pas encore écrit)  l écart global est chiffré, en points
  (pas encore écrit)  les tranches portent leur barre, pas seulement leur écart
  (pas encore écrit)  les tranches trop fines de VOTRE échantillon sont mises à part
  (pas encore écrit)  chaque tranche comparée porte son écart et son effectif
  (pas encore écrit)  les effectifs sont comptés
  (pas encore écrit)  la variable qui explique est nommée, et les deux médianes sont chiffrées
  (pas encore écrit)  l expérience qui trancherait est décrite en trois lignes


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [116]:
# TODO : trois hypothèses, ici.
# ── trois hypothèses, chacune avec ce qui la contredirait
hypothese("rdv_lointain",
          "un rendez-vous pris plus de 14 jours à l'avance s'oublie davantage : le taux d'absence monte (signe +)",
          "ajouter rdv_lointain = délai > 14 jours, et comparer au modèle de base sur la même validation",
          "le gain de score est inférieur à 2 barres de bruit, soit moins de 0,44 point sur la même validation")
hypothese("deja_absent",
          "un patient qui a déjà manqué un rendez-vous recommence plus souvent (signe +)",
          "ajouter deja_absent = taux_absence_passe > 0, et comparer au modèle de base",
          "le gain de score est inférieur à 2 barres de bruit, soit moins de 0,44 point sur la même validation")
hypothese("jeune_adulte",
          "les moins de 25 ans manquent plus souvent leurs rendez-vous (signe +)",
          "comparer le taux d'absence des moins de 25 ans à celui des autres, puis ajouter la colonne et mesurer",
          "l'écart de taux d'absence entre les deux groupes est inférieur à 2 barres, ou le score ne gagne pas 2 barres")

# ── le test de rdv_lointain : la prédiction AVANT
PRED_RDV = "mieux"   # mieux / pareil
prevoir("rdv_lointain", PRED_RDV, "les arbres coupent déjà sur delai_jours : la colonne apporte-t-elle du nouveau ?")

def mesure_rdv():
    a = Xtr.assign(rdv_lointain=(Xtr["delai_jours"] > 14).astype(int))
    b = Xval.assign(rdv_lointain=(Xval["delai_jours"] > 14).astype(int))
    return HistGradientBoostingClassifier(random_state=0).fit(a, ytr).score(b, yval)

s_rdv = essai("rdv_lointain", mesure_rdv, "ajout de rdv_lointain (délai > 14 jours)")
gain_rdv = (s_rdv - score_interne) / erreur_type
print(f"gain : {gain_rdv:+.2f} barre")

# ── le verdict, jugé sur le critère annoncé AVANT : 2 barres
confirmee_ou_contredite = "confirmée" if gain_rdv >= 2 else "contredite"
print("verdict :", confirmee_ou_contredite)

suite_donnee = ("Hypothèse contredite : rdv_lointain n'apporte aucun gain (+0,00 barre), parce que les arbres "
                "coupent déjà eux-mêmes delai_jours à n'importe quel seuil. Je l'abandonne. Une feature utile doit "
                "apporter une information que la table ne contient pas déjà, par exemple en combinant plusieurs colonnes, "
                "pas un simple seuil sur une colonne existante.")

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


,feature,ce qu'elle raconte,test,contredite si
0,rdv_lointain,un rendez-vous pris plus de 14 jours à l'avanc...,"ajouter rdv_lointain = délai > 14 jours, et co...",le gain de score est inférieur à 2 barres de b...


,feature,ce qu'elle raconte,test,contredite si
0,rdv_lointain,un rendez-vous pris plus de 14 jours à l'avanc...,"ajouter rdv_lointain = délai > 14 jours, et co...",le gain de score est inférieur à 2 barres de b...
1,deja_absent,un patient qui a déjà manqué un rendez-vous re...,"ajouter deja_absent = taux_absence_passe > 0, ...",le gain de score est inférieur à 2 barres de b...


,feature,ce qu'elle raconte,test,contredite si
0,rdv_lointain,un rendez-vous pris plus de 14 jours à l'avanc...,"ajouter rdv_lointain = délai > 14 jours, et co...",le gain de score est inférieur à 2 barres de b...
1,deja_absent,un patient qui a déjà manqué un rendez-vous re...,"ajouter deja_absent = taux_absence_passe > 0, ...",le gain de score est inférieur à 2 barres de b...
2,jeune_adulte,les moins de 25 ans manquent plus souvent leur...,comparer le taux d'absence des moins de 25 ans...,l'écart de taux d'absence entre les deux group...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,18:01:07
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",18:01:07
2,retirer 3 colonnes redondantes,pareil,une colonne constante et deux doublons n'appor...,18:01:21
3,arbres plus détaillés,mieux,63 feuilles par arbre au lieu de 31 : plus de ...,18:01:21
4,forêt aléatoire,pire,"une autre famille d'arbres, mais le plafond ob...",18:01:21
5,rdv_lointain,mieux,les arbres coupent déjà sur delai_jours : la c...,18:01:47


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 18:01:07"
2,retirer 3 colonnes redondantes,"sans zone_clinique, age_mois, cout_acte",pareil,82.51,"oui, à 18:01:21"
3,arbres plus détaillés,max_leaf_nodes = 63,mieux,82.50,"oui, à 18:01:21"
4,forêt aléatoire,"RandomForest, 100 arbres, sur la table d'indic...",pire,82.27,"oui, à 18:01:21"
5,rdv_lointain,ajout de rdv_lointain (délai > 14 jours),mieux,82.51,"oui, à 18:01:47"


gain : +0.00 barre
verdict : contredite
  ok        trois hypothèses
  ok        chacune dit ce qui la contredirait, précisément
  ok        une hypothèse a été testée sous son propre nom
  ok        le verdict est confirmée ou contredite
  ok        la suite donnée est écrite
  ok        les trois tableaux sont remplis avant le rapport


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [117]:
# TODO : une famille par colonne, du vocabulaire fermé ci-dessus.
familles = {
    "id_client": "identifiant",
    "date_defaut": "connue après l'issue",
    "statut_6_mois": "connue après l'issue",
    "montant": "utilisable",
    "canal": "risque de disparition",
    "score_interne": "utilisable"
}

# TODO : une phrase de justification par colonne.
justifications = {
    "id_client": "Elle désigne le client dans la base, mais ne dit rien de sa situation.",
    "date_defaut": "Au moment de la demande, on ne sait pas encore si l'emprunteur fera défaut.",
    "statut_6_mois": "Elle est relevée six mois après l'octroi, donc elle est inconnue au moment de décider.",
    "montant": "Le montant demandé est connu dès la demande.",
    "canal": "Il peut changer ou ne plus être alimenté en production.",
    "score_interne": "Il est calculé au moment de la demande, donc disponible pour décider.",
}

COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


  ok        les six colonnes sont classées
  ok        le vocabulaire est celui des quatre familles
  ok        la colonne du défaut n est pas classée « utilisable »
  ok        l identifiant est reconnu comme tel
  ok        le score calculé à la demande reste utilisable
  ok        les six justifications sont écrites, une phrase chacune


In [118]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,dossier_id sépare tout sans rien apprendre,aucun groupe de 500 lignes : 0 sur 89936 groupes,"écart de 100 points sans filtre, contre 28.9 p...","écarter : c'est un identifiant, une ligne par ..."
4,duree_consultation connaît la réponse,écart de 99.4 points entre ses groupes de 500 ...,"contre 28.9 points pour taux_absence_passe, la...",écarter : elle est remplie après le rendez-vou...


HYPOTHÈSES DE FEATURES 



,feature,ce qu'elle raconte,test,contredite si
0,rdv_lointain,un rendez-vous pris plus de 14 jours à l'avanc...,"ajouter rdv_lointain = délai > 14 jours, et co...",le gain de score est inférieur à 2 barres de b...
1,deja_absent,un patient qui a déjà manqué un rendez-vous re...,"ajouter deja_absent = taux_absence_passe > 0, ...",le gain de score est inférieur à 2 barres de b...
2,jeune_adulte,les moins de 25 ans manquent plus souvent leur...,comparer le taux d'absence des moins de 25 ans...,l'écart de taux d'absence entre les deux group...


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 18:01:07"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 18:01:07"
2,retirer 3 colonnes redondantes,"sans zone_clinique, age_mois, cout_acte",pareil,82.51,"oui, à 18:01:21"
3,arbres plus détaillés,max_leaf_nodes = 63,mieux,82.50,"oui, à 18:01:21"
4,forêt aléatoire,"RandomForest, 100 arbres, sur la table d'indic...",pire,82.27,"oui, à 18:01:21"
5,rdv_lointain,ajout de rdv_lointain (délai > 14 jours),mieux,82.51,"oui, à 18:01:47"


VOS CONTRÔLES 



,contrôle,tient
0,"la part d absents est un pourcentage, et c est...",True
1,"`ma_reference` est un pourcentage, au-dessus d...",True
2,les deux chiffres sont complémentaires,True
3,le prédicteur ne répond que des 0 et des 1,True
4,son score COÏNCIDE avec votre référence,True
...,...,...
72,le vocabulaire est celui des quatre familles,True
73,la colonne du défaut n est pas classée « utili...,True
74,l identifiant est reconnu comme tel,True
75,le score calculé à la demande reste utilisable,True


LA RELECTURE 

  ok        votre identifiant GitHub est renseigné
  ok        cinq constats
  ok        au moins deux constats mesurés par vous
  ok        chaque constat porte un chiffre ET une comparaison
  ok        trois hypothèses
  ok        chaque hypothèse dit ce qui la contredirait
  ok        au moins trois essais à vous — les amorces ne comptent pas
  ok        chaque essai porte un attendu — un signe ET une raison
  ok        les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle
  ok        une hypothèse a été testée sous son propre nom


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
